# Faz 11 portable Colab runner - Qwen embedding production only

This notebook is **not** a deployment of the institution system. It has one
job: produce pinned, hash-verified Qwen3-VL-Embedding-2B video/caption/query
embeddings from Google Drive-mounted data, shard-by-shard with resume, and
export NPY/Parquet/JSON artifacts you import into the real institution
FAZ11 stack (`python -m app.ingestion.ingest --dataset ...`) on a persistent
NVIDIA Linux + Docker Compose host.

**What Colab is for:** GPU embedding production, small-scale recall/latency
evaluation, artifact export.

**What Colab is *not* for:** a persistent PostgreSQL + ClickHouse + API + UI
deployment. Docker daemons are not a reliable production path inside a Colab
runtime. When the runtime disconnects or recycles, all *local* (non-Drive)
state is lost - only files actually written under Drive survive. Decoding
large videos directly from a mounted Drive path can be slow; this notebook
optionally copies each source video to local Colab SSD before decoding it,
then discards the local copy.

See `docs/COLAB_RUNBOOK.md` for the full explanation and the exact import
commands into the institution stack.

## 1. Environment checks (GPU, VRAM, disk) - run first, every session

In [ ]:
import subprocess
import shutil

import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU visible. In Colab: Runtime -> Change runtime type -> GPU, then restart and re-run."
    )
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
vram_total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print("GPU:", gpu_name, "compute capability:", capability, "VRAM: %.1f GB" % vram_total_gb)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

free_bytes, total_bytes = shutil.disk_usage("/content")
print("Local /content free disk: %.1f GB / %.1f GB" % (free_bytes / 1024**3, total_bytes / 1024**3))
if free_bytes < 10 * 1024**3:
    print("WARNING: less than 10 GB free on local disk; large videos should stay on Drive with per-file local staging only.")

## 2. Mount Drive and locate input/output roots

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/Multimodal-Video-Intelligence")
DATA_ROOT = DRIVE_ROOT / "data"           # institution videos/telemetry, same layout as a real DATA_ROOT
OUTPUT_ROOT = DRIVE_ROOT / "artifacts" / "colab_embeddings"
SHARD_ROOT = OUTPUT_ROOT / "shards"
SHARD_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_STAGE = Path("/content/stage")      # scratch copy of one video at a time; never the source of truth
LOCAL_STAGE.mkdir(parents=True, exist_ok=True)
print("Drive mounted:", DRIVE_ROOT.is_dir())
print("DATA_ROOT:", DATA_ROOT, "exists:", DATA_ROOT.is_dir())
print("OUTPUT_ROOT:", OUTPUT_ROOT)

## 3. Install pinned dependencies and fetch the pinned Qwen source/model

Uses the exact revisions pinned in `.env.example` / `docs/MODEL_BUNDLE.md`
for this repo state - not `main`, not "latest" - so embeddings produced here
carry the same `model_id`/`model_revision`/`source_commit` provenance the
institution stack's `verify_bundle()` expects.

In [ ]:
%pip install -q 'transformers>=4.57.3' 'accelerate>=1.12.0' 'qwen-vl-utils>=0.0.14' huggingface-hub pyarrow pandas numpy pillow

MODEL_ID = "Qwen/Qwen3-VL-Embedding-2B"
MODEL_REVISION = "9f2f7e710d6d81056aa5c0a4f04764fec6bb7bda"          # pinned - see docs/MODEL_BUNDLE.md
SOURCE_REPO = "https://github.com/QwenLM/Qwen3-VL-Embedding.git"
SOURCE_COMMIT = "393e2978d27852b0d0230d6994f37f9c15bed73c"           # pinned - see docs/MODEL_BUNDLE.md

import subprocess

repo_dir = Path("/content/Qwen3-VL-Embedding")
if not repo_dir.is_dir():
    subprocess.run(["git", "clone", SOURCE_REPO, str(repo_dir)], check=True)
subprocess.run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=repo_dir, check=True)
observed_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=repo_dir, capture_output=True, text=True, check=True
).stdout.strip()
assert observed_commit == SOURCE_COMMIT, f"source checkout mismatch: {observed_commit} != {SOURCE_COMMIT}"
print("Qwen source pinned at", observed_commit)

import sys
sys.path.insert(0, str(repo_dir))

In [ ]:
from huggingface_hub import model_info

from src.models.qwen3_vl_embedding import Qwen3VLEmbedder

info = model_info(MODEL_ID, revision=MODEL_REVISION)
assert info.sha == MODEL_REVISION or MODEL_REVISION in (info.sha, getattr(info, "id", None)), (
    f"model revision could not be confirmed against the hub: requested {MODEL_REVISION}"
)
dtype = torch.float16 if capability[0] < 8 else torch.bfloat16
embedder = Qwen3VLEmbedder(
    model_name_or_path=MODEL_ID, fps=1.0, max_frames=16, max_length=16384,
    torch_dtype=dtype, attn_implementation="sdpa",
)
print("Model pinned at revision", MODEL_REVISION, "dtype", dtype)

## 4. Discover shard plan and resume from previously completed shards

A shard is one batch of videos. Each shard's output files are written
atomically (temp name -> rename) so a crash mid-shard never leaves a
half-written shard that looks complete.

In [ ]:
import hashlib
import json as _json

SHARD_SIZE = 25  # videos per shard; tune to VRAM/session-length budget

video_paths = sorted((DATA_ROOT / "videos").rglob("*.mp4"))
if not video_paths:
    raise SystemExit(f"No videos found under {DATA_ROOT / 'videos'} - check the Drive mount and DATA_ROOT.")

shards = [video_paths[i:i + SHARD_SIZE] for i in range(0, len(video_paths), SHARD_SIZE)]
print(f"{len(video_paths)} videos -> {len(shards)} shards of up to {SHARD_SIZE}")

def shard_output_path(index: int) -> Path:
    return SHARD_ROOT / f"shard_{index:05d}.json"

completed_shards = {
    int(path.stem.split("_")[1])
    for path in SHARD_ROOT.glob("shard_*.json")
    if _json.loads(path.read_text(encoding="utf-8")).get("status") == "completed"
}
print(f"Already completed shards: {sorted(completed_shards)}")
pending_shard_indices = [i for i in range(len(shards)) if i not in completed_shards]
print(f"Resuming from {len(pending_shard_indices)} pending shard(s): {pending_shard_indices[:10]}{'...' if len(pending_shard_indices) > 10 else ''}")

## 5. Embed each pending shard (2048d base, MRL export to 1024/512/256)

In [ ]:
import shutil as _shutil
import time

import numpy as np

MRL_DIMENSIONS = (2048, 1024, 512, 256)

def _l2_normalize(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if np.any(norms <= 0):
        raise RuntimeError("zero-norm embedding vector produced")
    return vectors / norms

def _truncate_and_normalize(vectors: np.ndarray, dimension: int) -> np.ndarray:
    return _l2_normalize(vectors[:, :dimension].astype(np.float32, copy=False))

def embed_one_shard(index: int, videos: list[Path]) -> dict:
    started = time.perf_counter()
    local_copies = []
    ids, base_vectors = [], []
    for video_path in videos:
        local_path = LOCAL_STAGE / video_path.name
        _shutil.copyfile(video_path, local_path)
        local_copies.append(local_path)
        try:
            result = embedder.process([{"video": str(local_path)}])
            vector = result.detach().cpu().float().numpy()[0]
        finally:
            local_path.unlink(missing_ok=True)
        ids.append(video_path.relative_to(DATA_ROOT).as_posix())
        base_vectors.append(vector)
    base = np.stack(base_vectors).astype(np.float32)
    if base.shape[1] != 2048:
        raise RuntimeError(f"unexpected base embedding dimension: {base.shape}")
    if not np.isfinite(base).all():
        raise RuntimeError(f"shard {index}: non-finite values in base embeddings")
    if len(ids) != len(set(ids)):
        raise RuntimeError(f"shard {index}: duplicate video IDs within shard")

    per_dimension_paths = {}
    for dimension in MRL_DIMENSIONS:
        vectors = _truncate_and_normalize(base, dimension)
        out_path = SHARD_ROOT / f"shard_{index:05d}_dim{dimension}.npy"
        tmp_path = out_path.with_suffix(".npy.partial")
        np.save(tmp_path, vectors)
        tmp_path.replace(out_path)
        per_dimension_paths[str(dimension)] = out_path.name

    ids_path = SHARD_ROOT / f"shard_{index:05d}_ids.json"
    ids_tmp = ids_path.with_suffix(".json.partial")
    ids_tmp.write_text(_json.dumps(ids, indent=2), encoding="utf-8")
    ids_tmp.replace(ids_path)

    shard_hash = hashlib.sha256()
    for name in sorted(per_dimension_paths.values()) + [ids_path.name]:
        shard_hash.update((SHARD_ROOT / name).read_bytes())

    report = {
        "schema_version": 1, "shard_index": index, "video_count": len(ids),
        "status": "completed", "elapsed_s": round(time.perf_counter() - started, 3),
        "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "source_commit": SOURCE_COMMIT,
        "dimensions": per_dimension_paths, "ids_file": ids_path.name,
        "shard_sha256": shard_hash.hexdigest(),
    }
    report_path = shard_output_path(index)
    report_tmp = report_path.with_suffix(".json.partial")
    report_tmp.write_text(_json.dumps(report, indent=2), encoding="utf-8")
    report_tmp.replace(report_path)
    return report

for shard_index in pending_shard_indices:
    report = embed_one_shard(shard_index, shards[shard_index])
    print(f"shard {shard_index}: {report['video_count']} videos, sha256={report['shard_sha256'][:12]}..., {report['elapsed_s']}s")

## 6. Verify every shard and build the final export manifest

In [ ]:
all_reports = []
seen_ids = set()
for index in range(len(shards)):
    report_path = shard_output_path(index)
    if not report_path.is_file():
        raise RuntimeError(f"shard {index} was never completed - re-run section 5")
    report = _json.loads(report_path.read_text(encoding="utf-8"))
    if report.get("status") != "completed":
        raise RuntimeError(f"shard {index} status is {report.get('status')!r}, expected 'completed'")
    ids = _json.loads((SHARD_ROOT / report["ids_file"]).read_text(encoding="utf-8"))
    duplicate_with_prior = seen_ids.intersection(ids)
    if duplicate_with_prior:
        raise RuntimeError(f"duplicate video IDs across shards: {sorted(duplicate_with_prior)[:5]}")
    seen_ids.update(ids)
    for dimension_str, filename in report["dimensions"].items():
        vectors = np.load(SHARD_ROOT / filename)
        if vectors.shape[1] != int(dimension_str):
            raise RuntimeError(f"shard {index} dim {dimension_str}: shape {vectors.shape} mismatch")
        if not np.isfinite(vectors).all():
            raise RuntimeError(f"shard {index} dim {dimension_str}: non-finite vectors")
        norms = np.linalg.norm(vectors, axis=1)
        if not np.allclose(norms, 1.0, atol=1e-3):
            raise RuntimeError(f"shard {index} dim {dimension_str}: vectors are not L2-normalized")
    all_reports.append(report)

manifest = {
    "schema_version": 1, "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "source_commit": SOURCE_COMMIT, "shard_count": len(all_reports),
    "total_videos": len(seen_ids), "dimensions": list(MRL_DIMENSIONS),
    "shard_hashes": {report["shard_index"]: report["shard_sha256"] for report in all_reports},
}
manifest["manifest_sha256"] = hashlib.sha256(
    _json.dumps(manifest, sort_keys=True).encode("utf-8")
).hexdigest()
manifest_path = OUTPUT_ROOT / "embedding_manifest.json"
manifest_path.write_text(_json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print("All shards verified: no duplicates, all finite, all normalized, all dimensions correct.")
print("Final manifest:", manifest_path)
print(_json.dumps(manifest, indent=2, sort_keys=True))

## 7. Optional: small-scale recall/latency evaluation

Runs a tiny in-memory exact cosine search over the exported base (2048d)
vectors against a handful of held-out query embeddings you provide - a
sanity check, not a substitute for the institution stack's own benchmark
matrix (`docs/BLOCKERS.md` Faz 7-8 notes).

In [ ]:
def small_scale_eval(query_texts: list[str], k: int = 5):
    base_vectors, ids = [], []
    for index in range(len(shards)):
        report = _json.loads(shard_output_path(index).read_text(encoding="utf-8"))
        base_vectors.append(np.load(SHARD_ROOT / report["dimensions"]["2048"]))
        ids.extend(_json.loads((SHARD_ROOT / report["ids_file"]).read_text(encoding="utf-8")))
    corpus = np.concatenate(base_vectors, axis=0)
    results = {}
    for text in query_texts:
        query_result = embedder.process([{"text": text}])
        query_vector = _l2_normalize(query_result.detach().cpu().float().numpy())[0]
        scores = corpus @ query_vector
        top_k = np.argsort(-scores)[:k]
        results[text] = [(ids[i], float(scores[i])) for i in top_k]
    return results

# Example (uncomment and edit with real held-out queries before running):
# print(small_scale_eval(["low altitude flight over water"]))

## 8. Export and import into the institution stack

`OUTPUT_ROOT` on Drive now contains `shards/shard_*_dim{2048,1024,512,256}.npy`,
`shards/shard_*_ids.json`, and `embedding_manifest.json`. Download this
directory (zip it from Drive, or `rclone`/`gsutil` it) to the institution
host's `DATA_ROOT`-adjacent staging area, then run the real ingest against
the same dataset manifest used here - the institution stack re-embeds
nothing; it is the source of truth for search, not this notebook. See
`docs/COLAB_RUNBOOK.md` for the exact import commands.

In [ ]:
import shutil as _shutil2

zip_path = DRIVE_ROOT / "artifacts" / "colab_embeddings_export"
_shutil2.make_archive(str(zip_path), "zip", root_dir=OUTPUT_ROOT)
print("Exported:", f"{zip_path}.zip")
print("Next: copy this zip to the institution host and follow docs/COLAB_RUNBOOK.md's import section.")